In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Data splitting and validation
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Feature selection (Methodology 3.4 - Feature Selection)
from sklearn.feature_selection import SelectFromModel


In [11]:
df = pd.read_excel('df_final.xlsx')  # Load your dataset here

In [12]:
df = df.copy()
# TARGET VARIABLE

df["At_Risk_Status"] = df["At_Risk_Status"].map({
    "No": 0,
    "Yes": 1
})

# REMOVE IDENTIFIER
df = df.drop(columns=["Student_ID"], errors="ignore")

# DEFINE X AND y

target = "At_Risk_Status"

X = df.drop(columns=[target])
y = df[target]

# CHECK

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nData types:")
print(X.dtypes)

X shape: (330, 21)
y shape: (330,)

Target distribution:
At_Risk_Status
0    209
1    121
Name: count, dtype: int64

Data types:
Academic Year                                int64
APPLIED MECHANICS                            int64
COMPUTER SKILLS                              int64
CONSTRUCTION MATERIALS TECHNOLOGY            int64
ENGINEERING CHEMISTRY                        int64
ENGINEERING DRAWING 1                        int64
ENGINEERING MATHEMATICS 1                    int64
ENGINEERING PHYSICS                          int64
PROFESSIONAL AND TECHNICAL COMMUNICATION     int64
SURVEYING                                    int64
THEORY OF STRUCTURES                         int64
Gender                                      object
Language Name                               object
Ethnic Group Name                           object
Previous Activity Desc                      object
Matric Type                                 object
Age_At_Enrolment                             int64
Burs

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Training target distribution:")
print(y_train.value_counts(normalize=True))
print(f"\nTest target distribution:")
print(y_test.value_counts(normalize=True))


Training set: 231 samples
Test set: 99 samples
Training target distribution:
At_Risk_Status
0    0.632035
1    0.367965
Name: proportion, dtype: float64

Test target distribution:
At_Risk_Status
0    0.636364
1    0.363636
Name: proportion, dtype: float64


In [14]:
categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

In [15]:
print("categorical_features:", categorical_features)

print("numerical_features:", numerical_features)

categorical_features: ['Gender', 'Language Name', 'Ethnic Group Name', 'Previous Activity Desc', 'Matric Type', 'Bursary Type', 'Funded', 'Residence_Status', 'Secondary_School_Quintile']
numerical_features: ['Academic Year', 'APPLIED MECHANICS', 'COMPUTER SKILLS', 'CONSTRUCTION MATERIALS TECHNOLOGY', 'ENGINEERING CHEMISTRY', 'ENGINEERING DRAWING 1', 'ENGINEERING MATHEMATICS 1', 'ENGINEERING PHYSICS', 'PROFESSIONAL AND TECHNICAL COMMUNICATION', 'SURVEYING', 'THEORY OF STRUCTURES', 'Age_At_Enrolment']


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer

preprocessor = ColumnTransformer(
    transformers=[

        # Numerical variables
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numerical_features
        ),

        # Categorical variables
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                # Some categorical columns (e.g. Secondary_School_Quintile) mix
                # numeric and string values (e.g. 1, 2, "Not applicable"). Casting
                # everything to string after imputation avoids the
                # "must be uniformly strings or numbers" error OneHotEncoder
                # raises when it tries to sort a mix of int and str categories.
                ("to_str", FunctionTransformer(lambda X: X.astype(str), feature_names_out="one-to-one")),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ))
            ]),
            categorical_features
        )
    ]
)


Feature Selection

In [17]:
X_train_fs = fs_preprocessor.fit_transform(X_train, y_train)

TypeError: Encoders require their input argument must be uniformly strings or numbers. Got ['int', 'str']

In [ ]:
# Feature Selection (Methodology 3.4)
# Fit the preprocessing pipeline on the training data only, then use a
# Random Forest's feature importances to select the most relevant features
# and flag redundant/low-importance ones, reducing noise for the models
# fitted below.

fs_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numerical_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            # Same fix as the main preprocessor above: force categorical
            # columns to a uniform string type before one-hot encoding, since
            # a mix of int/str values otherwise breaks OneHotEncoder's
            # automatic category detection.
            ("to_str", FunctionTransformer(lambda X: X.astype(str), feature_names_out="one-to-one")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_features)
    ]
)

X_train_fs = fs_preprocessor.fit_transform(X_train, y_train)
fs_feature_names = fs_preprocessor.get_feature_names_out()

fs_estimator = RandomForestClassifier(
    n_estimators=300, random_state=42, class_weight="balanced", n_jobs=-1
)
fs_estimator.fit(X_train_fs, y_train)

selector = SelectFromModel(fs_estimator, threshold="median", prefit=True)
selected_mask = selector.get_support()

feature_selection_df = pd.DataFrame({
    "Feature": fs_feature_names,
    "Importance": fs_estimator.feature_importances_,
    "Selected": selected_mask
}).sort_values("Importance", ascending=False)

print(f"Total features: {len(fs_feature_names)}")
print(f"Selected (above-median importance): {selected_mask.sum()}")
feature_selection_df.head(20)


Testing Logistic Regression, Random Forest

In [ ]:
# 10-fold stratified cross-validation, used throughout for model
# validation (LeaveOneOut was replaced: it is computationally impractical
# inside a GridSearchCV and was inconsistent with the "10-Fold CV" labels
# used further down).
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

Logistic Regression

In [ ]:
# LOGISTIC REGRESSION

from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),

    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

In [ ]:
# LOGISTIC REGRESSION - 10-FOLD CV

logistic_cv = cross_validate(
    estimator=logistic_model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("\nLOGISTIC REGRESSION - 10-FOLD CV")

for metric in scoring:

    scores = logistic_cv[f"test_{metric}"]

    print(
        f"{metric.capitalize():12}: "
        f"{scores.mean():.4f} "
        f"(+/- {scores.std():.4f})"
    )


LOGISTIC REGRESSION - 10-FOLD CV
Accuracy    : 0.8524 (+/- 0.0854)
Precision   : 0.7853 (+/- 0.1051)
Recall      : 0.8306 (+/- 0.1599)
F1          : 0.8001 (+/- 0.1200)
Roc_auc     : 0.9257 (+/- 0.0702)


In [ ]:
# LOGISTIC REGRESSION - FINAL MODEL

logistic_model.fit(
    X_train,
    y_train
)

logistic_pred = logistic_model.predict(X_test)

logistic_prob = logistic_model.predict_proba(
    X_test
)[:, 1]

In [ ]:
print("\nLOGISTIC REGRESSION - TEST SET")

print(classification_report(
    y_test,
    logistic_pred,
    target_names=["Not At Risk", "At Risk"]
))

print("Accuracy:",
      accuracy_score(y_test, logistic_pred))

print("Precision:",
      precision_score(y_test, logistic_pred))

print("Recall:",
      recall_score(y_test, logistic_pred))

print("F1:",
      f1_score(y_test, logistic_pred))

print("ROC AUC:",
      roc_auc_score(y_test, logistic_prob))


LOGISTIC REGRESSION - TEST SET
              precision    recall  f1-score   support

 Not At Risk       0.83      0.87      0.85        63
     At Risk       0.76      0.70      0.73        37

    accuracy                           0.81       100
   macro avg       0.80      0.79      0.79       100
weighted avg       0.81      0.81      0.81       100

Accuracy: 0.81
Precision: 0.7647058823529411
Recall: 0.7027027027027027
F1: 0.7323943661971831
ROC AUC: 0.9069069069069069


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# PREPARE DATA FOR STATSMODELS LOGISTIC REGRESSION

# X_train_sm / sm_feature_names were referenced here but never created -
# that's what raised the NameError. Reuse the already-fitted preprocessor
# from the logistic_model pipeline (fit in the "FINAL MODEL" cell above)
# to build the transformed training matrix and its feature names.
X_train_sm = logistic_model.named_steps["preprocessor"].transform(X_train)
sm_feature_names = logistic_model.named_steps["preprocessor"].get_feature_names_out()

# Convert to DataFrame
X_train_sm_df = pd.DataFrame(
    X_train_sm,
    columns=sm_feature_names
)

# Make sure everything is numeric
X_train_sm_df = X_train_sm_df.apply(pd.to_numeric, errors="coerce")

# Replace infinite values
X_train_sm_df = X_train_sm_df.replace([np.inf, -np.inf], np.nan)

# Remove columns containing NaN
X_train_sm_df = X_train_sm_df.dropna(axis=1)

# REMOVE CONSTANT COLUMNS

constant_cols = [
    col for col in X_train_sm_df.columns
    if X_train_sm_df[col].nunique() <= 1
]

if constant_cols:
    print("Removing constant columns:")
    print(constant_cols)

    X_train_sm_df = X_train_sm_df.drop(columns=constant_cols)

# REMOVE DUPLICATE COLUMNS

duplicate_cols = X_train_sm_df.T.duplicated()

if duplicate_cols.any():
    duplicate_names = X_train_sm_df.columns[duplicate_cols].tolist()

    print("\nRemoving duplicate columns:")
    print(duplicate_names)

    X_train_sm_df = X_train_sm_df.loc[:, ~duplicate_cols]

# CHECK FOR PERFECT MULTICOLLINEARITY

# Add constant/intercept
X_train_sm_df = sm.add_constant(
    X_train_sm_df,
    has_constant="add"
)

# Target
y_train_sm = pd.Series(y_train).reset_index(drop=True)

# Make sure X and y have the same number of rows
X_train_sm_df = X_train_sm_df.reset_index(drop=True)

print("\nFinal X shape:", X_train_sm_df.shape)
print("Final y shape:", y_train_sm.shape)

# FIT LOGISTIC REGRESSION

try:

    logit_result = sm.Logit(
        y_train_sm,
        X_train_sm_df
    ).fit(
        method="lbfgs",
        maxiter=1000,
        disp=0
    )

    print(logit_result.summary())

except np.linalg.LinAlgError:

    print("\nSingular matrix detected.")
    print("Trying regularized logistic regression...")

    logit_result = sm.Logit(
        y_train_sm,
        X_train_sm_df
    ).fit_regularized(
        method="l1",
        alpha=0.01,
        maxiter=2000,
        disp=0
    )

    print(logit_result.summary())

Random Forest

In [ ]:
# RANDOM FOREST — grid search (10-fold CV) + final model
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, class_weight="balanced", n_jobs=-1))
])

param_grid = {
    "classifier__n_estimators": [100, 300, 500],
    "classifier__max_depth": [None, 5, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 5]
}

grid_search = GridSearchCV(
    rf_model,
    param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

rf_model = grid_search.best_estimator_
print("Best RF params:", grid_search.best_params_)

pred = rf_model.predict(X_test)
prob = rf_model.predict_proba(X_test)[:, 1]

print("\nRANDOM FOREST - TEST SET")
print(classification_report(y_test, pred, target_names=["Not At Risk", "At Risk"]))

print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, prob))

# 10-FOLD CROSS VALIDATION (on the training set, using the tuned model)
cv_results = cross_validate(
    estimator=rf_model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("\nRandom Forest (10-Fold CV)")
for metric in scoring.keys():
    scores = cv_results[f"test_{metric}"]
    print(f"{metric.capitalize():10}: {scores.mean():.4f} (+/- {scores.std():.4f})")


In [ ]:
import seaborn as sns

cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Not At Risk", "At Risk"],
            yticklabels=["Not At Risk", "At Risk"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest — Confusion Matrix")
plt.show()

In [ ]:
import pandas as pd

# Get feature names after preprocessing
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()

# Get feature importances
importances = rf_model.named_steps["classifier"].feature_importances_

# Create table
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

# Display Top 20
importance_df.head(20)

In [ ]:
import matplotlib.pyplot as plt

top20 = importance_df.head(20)

plt.figure(figsize=(10,8))

plt.barh(
    top20["Feature"],
    top20["Importance"],
    color="forestgreen"
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top 20 Most Important Features (Random Forest)")

plt.gca().invert_yaxis()

# Add values
for i, value in enumerate(top20["Importance"]):
    plt.text(value, i, f"{value:.3f}", va="center")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Extract one tree
tree = rf_model.named_steps["classifier"].estimators_[0]

plt.figure(figsize=(30,15))

plot_tree(
    tree,
    feature_names=feature_names,
    class_names=["Not At Risk", "At Risk"],
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3      # keeps the tree readable
)

plt.title("Random Forest - Tree 1")
plt.show()

In [ ]:
lr_clf = logistic_model.named_steps["classifier"]
lr_feature_names = logistic_model.named_steps["preprocessor"].get_feature_names_out()

coef_df = pd.DataFrame({
    "Feature": lr_feature_names,
    "Coefficient": lr_clf.coef_[0]
})
coef_df["Odds Ratio"] = np.exp(coef_df["Coefficient"])
coef_df["Direction"] = np.where(
    coef_df["Coefficient"] > 0, "Associated with At-Risk", "Associated with Not-At-Risk"
)
coef_df = coef_df.reindex(coef_df["Coefficient"].abs().sort_values(ascending=False).index)

top25 = coef_df.head(25)
colors = ["crimson" if v > 0 else "steelblue" for v in top25["Coefficient"]]

plt.figure(figsize=(10, 9))
plt.barh(top25["Feature"], top25["Coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Logistic Regression Coefficient (standardised features)")
plt.title("Direction & Strength of Association with At-Risk Status\n"
          "(red = associated with At-Risk, blue = associated with Not-At-Risk)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# NOTE: categories are one-hot encoded without dropping a reference level,
# so read each dummy's coefficient as "this category vs NOT being in it"
# under L2 shrinkage, not a clean vs-baseline odds ratio. For textbook
# reference-category odds ratios, refit with OneHotEncoder(drop="first")
# or use statsmodels.

coef_df.head(25)


In [ ]:
# Example the supervisor/reader will likely ask about first: Age_At_Enrolment.
print(coef_df[coef_df["Feature"].str.contains("Age_At_Enrolment")].to_string(index=False))


### Random Forest direction (SHAP)

In [ ]:
!pip install shap

In [ ]:
# SHAP gives the Random Forest the same kind of directional reading the
# Logistic Regression coefficients give above: for every prediction, which
# way did each feature push it, and by how much.
# Requires: pip install shap
try:
    import shap

    rf_clf = rf_model.named_steps["classifier"]
    rf_preprocessor = rf_model.named_steps["preprocessor"]
    X_test_transformed = rf_preprocessor.transform(X_test)
    rf_feature_names = rf_preprocessor.get_feature_names_out()

    explainer = shap.TreeExplainer(rf_clf)
    shap_values = explainer.shap_values(X_test_transformed)
    # Newer shap versions return an Explanation object or a 3-D array
    # (samples, features, classes); older ones return a list [class0, class1].
    if isinstance(shap_values, list):
        sv = shap_values[1]
    elif hasattr(shap_values, "values") and shap_values.values.ndim == 3:
        sv = shap_values.values[:, :, 1]
    else:
        sv = shap_values

    shap.summary_plot(
        sv, X_test_transformed, feature_names=rf_feature_names, show=True
    )
    # Positive SHAP value = pushes THIS prediction toward At-Risk (class 1).
    # Point colour = the feature's own value (red = high, blue = low), so
    # e.g. a cluster of red dots on the positive side for Age_At_Enrolment
    # would mean "older students are pushed toward At-Risk".
except ImportError:
    print("SHAP is not installed in this environment.")
    print("Run: pip install shap")
    print("Then re-run this cell to see which way (not just how much) each")
    print("feature pushes the Random Forest's predictions.")


In [ ]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, prob)

plt.figure(figsize=(8, 5))
plt.plot(thresholds, precisions[:-1], label="Precision")
plt.plot(thresholds, recalls[:-1], label="Recall")
plt.axvline(0.5, color="gray", linestyle="--", linewidth=1, label="Default threshold (0.5)")
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Precision-Recall Trade-off vs Threshold — Random Forest (Experiment 1)")
plt.legend()
plt.tight_layout()
plt.show()

target_recall = 0.85
valid = [(t, p, r) for p, r, t in zip(precisions[:-1], recalls[:-1], thresholds) if r >= target_recall]
if valid:
    best_threshold, best_precision, best_recall = max(valid, key=lambda x: x[0])
    print(f"Lowest threshold reaching >= {target_recall:.0%} recall: {best_threshold:.3f}")
    print(f"  -> Recall: {best_recall:.3f}, Precision: {best_precision:.3f}")

    pred_tuned = (prob >= best_threshold).astype(int)
    print("\nClassification report at tuned threshold:")
    print(classification_report(y_test, pred_tuned, target_names=["Not At Risk", "At Risk"]))
else:
    print(f"No threshold in this test set reaches {target_recall:.0%} recall.")
